# Massachusetts General Hospital

## Collaborative effort to investigate the plasma proteomic signatures of COVID-19 positive patients

*Data provided by the MGH Emergency Department COVID-19 Cohort (Filbin, Goldberg, Hacohen) with Olink Proteomics.* https://www.olink.com/mgh-covid-study/

### Initial analysis

#### Plasma proteomics reveals tissue-specific cell death and mediators of cell-cell interactions in severe COVID-19 patients

https://www.biorxiv.org/content/10.1101/2020.11.02.365536v2

By analyzing several thousand plasma proteins in 306 COVID-19 patients and 78
symptomatic controls over serial timepoints using two complementary approaches, we uncover COVID-19
host immune and non-immune proteins not previously linked to this disease. Integration of plasma proteomics
with nine published scRNAseq datasets shows that SARS-CoV-2 infection upregulates
monocyte/macrophage, plasmablast, and T cell effector proteins. By comparing patients who died to severely
ill patients who survived, we identify dynamic immunomodulatory and tissue-associated proteins associated
with survival, providing insights into which host responses are beneficial and which are detrimental to survival. 


![Study design](https://www.olink.com/content/uploads/2020/09/MGH-Study-outline-final.png "Massachusetts General Hospital COVID-19")




### Proposed Analyses

1. We reproduce the results obtained in Filbin et al 2020 using the Olink data (timepoint D0)

2. We compare COVID-19 positive patients to identify biomarkers of severity (timepoint D0)

*Note: To define the different severity groups we make use of the WHO scores assigned to the patients.*


##### Load CKG libraries

In [ ]:
import os

from ckg.analytics_core.analytics import analytics
from ckg.analytics_core.viz import viz

from ckg.graphdb_connector import connector
from ckg.graphdb_builder import builder_utils

from plotly.offline import init_notebook_mode, iplot
%matplotlib inline
init_notebook_mode(connected=True)

## Reading proteomics and clinical data

In [ ]:
#### Access: Data provided by the MGH Emergency Department COVID-19 Cohort
#### (Filbin, Goldberg, Hacohen) with Olink Proteomics. https://www.olink.com/mgh-covid-study/

olink_file = '../../assets/MGH_COVID_OLINK_NPX.txt'
olink_clin_file = '../../assets/MGH_COVID_Clinical_Info.txt'

### Proteomics data

In [ ]:
olink_data = builder_utils.readDataFromCSV(olink_file, sep=';', comment='#')

In [ ]:
olink_data.head()

#### QC and data exploration

In [ ]:
print("Number of proteins:", len(olink_data['UniProt'].unique()))

In [ ]:
olink_data.shape

##### Remove proteins with QC warnings

In [ ]:
olink_data['QC_Warning'].unique()

In [ ]:
olink_data.groupby('QC_Warning').count()['SampleID']

In [ ]:
olink_data = olink_data[olink_data['QC_Warning'] == 'Pass']

In [ ]:
olink_data.shape

In [ ]:
print("Number of proteins that passed Olink QC:", len(olink_data['UniProt'].unique()))

In [ ]:
print("Total number of patients:", len(olink_data['subject_id'].unique()))

In [ ]:
print("Total number of samples:", len(olink_data['SampleID'].unique()))

In [ ]:
olink_data.shape

##### Remove rows with missing subject id

In [ ]:
olink_data = olink_data.dropna(subset=['subject_id'])

In [ ]:
olink_data['subject_id'] = olink_data['subject_id'].astype('int').astype('str')

In [ ]:
olink_data.shape

##### Merge protein names and identifiers

In [ ]:
olink_data['identifier'] = olink_data['Assay'] +"~"+olink_data['UniProt']

In [ ]:
olink_data.head()

### Clinical data

In [ ]:
olink_clin_data = builder_utils.readDataFromCSV(olink_clin_file, sep=';', comment='#')

In [ ]:
olink_clin_data.head()

In [ ]:
plot = viz.get_boxplot_grid(olink_clin_data, identifier="clin_vars", args={"title": "Clinical variables",
                                                                           "x":"COVID", 
                                                                           "y":"Age cat",
                                                                           "color":"COVID", "width":600})
iplot(plot.figure)

In [ ]:
olink_clin_data['subject_id'] = olink_clin_data['subject_id'].astype('str')

In [ ]:
olink_clin_data.columns.tolist()

#### Merging both datasets

In [ ]:
data = olink_data.set_index('subject_id').join(olink_clin_data.set_index('subject_id')).reset_index()

In [ ]:
data.head()


In [ ]:
data.shape

## 1. Covid-19 positive vs negative

##### Study timepoint D0

In [ ]:
data = data[data['Timepoint'] == "D0"]

In [ ]:
data.shape

In [ ]:
print("Total number of positive patients:", len(data['subject_id'].unique()))

In [ ]:
df = data[['subject_id', 'SampleID', 'identifier', 'WHO max', 'WHO 0', 'NPX', 'Age cat', 'COVID', 'BMI cat', 'Timepoint', 'HEART']]

In [ ]:
print("Total number of positive patients:", len(data['subject_id'].unique()))

##### Transform into wide format

In [ ]:
df_wide = analytics.transform_into_wide_format(df, index=['SampleID', 'subject_id'], columns=['identifier'], values='NPX', extra=['WHO max', 'WHO 0', 'Age cat', 'COVID', 'BMI cat','HEART'])

In [ ]:
df_wide.head()

In [ ]:
df_wide.shape

In [ ]:
df_wide.describe()

In [ ]:
result = analytics.run_correlation(df_wide, alpha=0.05, subject='subject_id', group="COVID", method="spearman", correction='fdr_bh')

In [ ]:
net = viz.get_network(result, identifier='corr_net', args={'source':'node1', 'target':'node2', 
                                                     'weight':'weight', 'values':'weight', 
                                                           'title':'Correlation Network',
                                                        'color_weight': True, 'node_size':'degree', 
                                                           'cutoff': 0.5, 'cutoff_abs':True})

In [ ]:
viz.visualize_notebook_network(net['notebook'])

##### Save dataset to file

In [ ]:
df_wide.to_csv('olink_data_NPX_values_WHO_max.tsv', sep='\t', index=False, header=True, doublequote=False)

##### Processing:

- Imputation using mixed model: KNN when 60% valid values or Probabilistic Minimum Imputation otherwise
- Imputation per group based on WHO 0


In [ ]:
df_wide = analytics.imputation_mixed_norm_KNN(df_wide, index_cols=['WHO 0', 'WHO max', 'SampleID', 'subject_id', 'Age cat', 'COVID', 'BMI cat', 'HEART'], shift=1.8, nstd=0.3, group='COVID', cutoff=0.6)

In [ ]:
df_wide.head()

In [ ]:
df_wide.to_csv('olink_data_NPX_values_imputed_KNN.tsv', sep='\t', index=False, header=True, doublequote=False)

In [ ]:
df_wide = df_wide.reset_index()

In [ ]:
df_wide['COVID'] = ['COVID-19 positive' if c else 'COVID-19 negative' for c in df_wide['COVID']]

In [ ]:
pca, args = analytics.run_pca(df_wide, drop_cols=['SampleID', 'subject_id', 'Age cat', 'WHO max', 'BMI cat', 'HEART'], group='COVID', annotation_cols=['SampleID'])
args['group'] = 'group'
args['hovering_cols'] = ['SampleID']
args['factor'] = 250
args['loadings'] = 15
args['title'] = 'Olink data'

In [ ]:
figure = viz.get_pca_plot(pca, identifier='pca', args=args)
iplot(figure.figure)

In [ ]:
pca, args = analytics.run_pca(df_wide, drop_cols=['SampleID', 'subject_id', 'Age cat', 'COVID', 'BMI cat', 'HEART', 'WHO max'], group='WHO 0', annotation_cols=['SampleID'])
args['group'] = 'group'
args['hovering_cols'] = ['SampleID']
args['factor'] = 250
args['loadings'] = 15
args['title'] = 'Olink data coloured by WHO score at timepoint D0'

In [ ]:
figure = viz.get_pca_plot(pca, identifier='pca', args=args)
iplot(figure.figure)

In [ ]:
pca, args = analytics.run_pca(df_wide, drop_cols=['SampleID', 'subject_id', 'Age cat', 'COVID', 'BMI cat', 'HEART', 'WHO 0'], group='WHO max', annotation_cols=['SampleID'])
args['group'] = 'group'
args['hovering_cols'] = ['SampleID']
args['factor'] = 250
args['loadings'] = 15
args['title'] = 'Olink data coloured by maximum WHO score'

In [ ]:
figure = viz.get_pca_plot(pca, identifier='pca', args=args)
iplot(figure.figure)

In [ ]:
pca, args = analytics.run_pca(df_wide, drop_cols=['SampleID', 'subject_id', 'WHO max', 'COVID', 'BMI cat', 'HEART'], group='Age cat', annotation_cols=['SampleID'])
args['group'] = 'group'
args['hovering_cols'] = ['SampleID']
args['factor'] = 250
args['loadings'] = 15
args['title'] = 'Olink data coloured by age category'

In [ ]:
figure = viz.get_pca_plot(pca, identifier='pca', args=args)
iplot(figure.figure)

In [ ]:
pca, args = analytics.run_pca(df_wide, drop_cols=['SampleID', 'subject_id', 'WHO max', 'COVID', 'BMI cat'], group='HEART', annotation_cols=['SampleID'])
args['group'] = 'group'
args['hovering_cols'] = ['SampleID']
args['factor'] = 350
args['loadings'] = 20
args['title'] = 'Olink data coloured by HEART'

In [ ]:
figure = viz.get_pca_plot(pca, identifier='pca', args=args)
iplot(figure.figure)

In [ ]:
pca, args = analytics.run_pca(df_wide, drop_cols=['SampleID', 'subject_id', 'WHO max', 'COVID', 'HEART'], group='BMI cat', annotation_cols=['SampleID'])
args['group'] = 'group'
args['hovering_cols'] = ['SampleID']
args['factor'] = 350
args['loadings'] = 20
args['title'] = 'Olink data coloured by BMI cat'

In [ ]:
figure = viz.get_pca_plot(pca, identifier='pca', args=args)
iplot(figure.figure)

## Differential regulation: Covid-19 positive vs negative

In [ ]:
ancova_results = analytics.run_ancova(df_wide, covariates=['Age cat', 'HEART', 'BMI cat'], drop_cols=['SampleID', 'subject_id', 'WHO max'], subject='subject_id', group='COVID', is_logged=True)

In [ ]:
volcano_plot = viz.run_volcano(ancova_results, identifier='proteomics_volcanos', 
            args={'alpha':0.01, 'fc':2, 'num_annotations': 50,
                  'colorscale':'Blues', 'showscale': False, 
                  'marker_size':8, 'x_title':'log2FC', 'y_title':'-log10(pvalue)'})

for plot in volcano_plot:
    iplot(plot.figure)

## Functional enrichment

In [ ]:
go_terms_query = "MATCH (p:Protein)-[]-(bp:Biological_process) WHERE (p.name+'~'+p.id) IN {} RETURN DISTINCT (p.name+'~'+p.id) AS identifier,bp.name AS annotation"
go_terms_query = go_terms_query.format(df_wide.columns.tolist())
annotation = connector.run_query(go_terms_query)

In [ ]:
annotation.head()

In [ ]:
annotation.shape

In [ ]:
enrichment_results = analytics.run_up_down_regulation_enrichment(ancova_results, annotation, identifier='identifier', groups=['group1', 'group2'], annotation_col='annotation', reject_col='rejected', group_col='group', method='fisher', correction='fdr_bh', alpha=0.01, lfc_cutoff=1)

In [ ]:
figures = viz.get_enrichment_plots(enrichment_results, identifier='enrichment', args={'width':1800})
for fig in figures:
    iplot(fig.figure)

In [ ]:
df_wide = df_wide[df_wide['COVID'] == 'COVID-19 positive']

In [ ]:
df_wide['WHO 0'] = df_wide['WHO 0'].astype('str')
df_wide['WHO max'] = df_wide['WHO max'].astype('str')

In [ ]:
df_wide.shape

In [ ]:
ancova_results = analytics.run_ancova(df_wide, covariates=['Age cat', 'HEART', 'BMI cat'], drop_cols=['SampleID', 'subject_id', 'COVID', 'WHO 0'], subject='subject_id', group='WHO max', is_logged=True)

In [ ]:
volcano_plot = viz.run_volcano(ancova_results, identifier='proteomics_volcanos', 
            args={'alpha':0.01, 'fc':2, 'num_annotations': 50,
                  'colorscale':'Blues', 'showscale': False, 
                  'marker_size':8, 'x_title':'log2FC', 'y_title':'-log10(pvalue)'})

for plot in volcano_plot:
    iplot(plot.figure)

In [ ]:
go_terms_query = "MATCH (p:Protein)-[]-(bp:Biological_process) WHERE (p.name+'~'+p.id) IN {} RETURN DISTINCT (p.name+'~'+p.id) AS identifier,bp.name AS annotation"
go_terms_query = go_terms_query.format(df_wide.columns.tolist())
annotation = connector.run_query(go_terms_query)

In [ ]:
annotation.head()

In [ ]:
annotation.shape

In [ ]:
enrichment_results = analytics.run_up_down_regulation_enrichment(ancova_results, annotation, identifier='identifier', groups=['group1', 'group2'], annotation_col='annotation', reject_col='rejected', group_col='group', method='fisher', correction='fdr_bh', alpha=0.01, lfc_cutoff=1)

In [ ]:
figures = viz.get_enrichment_plots(enrichment_results, identifier='enrichment', args={'width':1800})
for fig in figures:
    iplot(fig.figure)